# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  window                   120
  target_delay             13675
  initial_time             2000-07-27T14:57:22
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           12969 frames  (216.15s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [2]:
x.precompute_chart()
x.save()

[expedition] 13:28:05  === precompute_chart ===  (2026-09-08)
[expedition] 13:28:05  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  workers=12
[expedition] 13:28:06  mdmsh 1/248  elapsed 0.0m  eta ~4.8m
[expedition] 13:28:06  mdmsh 5/248  elapsed 0.0m  eta ~0.9m
[expedition] 13:28:06  mdmsh 10/248  elapsed 0.0m  eta ~0.5m
[expedition] 13:29:00  mdmsh 15/248  elapsed 0.9m  eta ~14.0m
[expedition] 13:30:13  mdmsh 20/248  elapsed 2.1m  eta ~24.2m
[expedition] 13:31:23  mdmsh 25/248  elapsed 3.3m  eta ~29.4m
[expedition] 13:32:34  mdmsh 30/248  elapsed 4.5m  eta ~32.5m
[expedition] 13:33:44  mdmsh 35/248  elapsed 5.6m  eta ~34.3m
[expedition] 13:34:33  mdmsh 40/248  elapsed 6.5m  eta ~33.6m
[expedition] 13:35:34  mdmsh 45/248  elapsed 7.5m  eta ~33.7m
[expedition] 13:36:41  mdmsh 50/248  elapsed 8.6m  eta ~34.0m
[expedition] 13:37:49  mdmsh 55/248  elapsed 9.7m  eta ~34.1m
[expedition] 13:38:53  mdmsh 60/248  elapsed

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [6]:
x.chart_report()
x.save()

[expedition] 15:13:12  === chart_report ===
Best (boot time, M) pairs  [top 10 of 1666]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-05-30 14:59:59     481487       29263     477       29.9%    88.5
   2  2000-01-01 14:00:11     553868       33607     549       28.9%    94.9
   3  2000-01-01 14:00:11     292003       17891     287       28.6%    68.9
   4  2000-07-29 14:56:09     292803       17939     288       28.4%    69.0
   5  2000-05-31 14:56:57     245948       15127     241       28.3%    63.3
   6  2000-07-29 14:55:10     353587       21587     349       28.2%    75.9
   7  2000-01-01 14:01:10     294336       18031     289       28.2%    69.2
   8  2000-01-01 14:00:11     412638       25131     408       28.1%    81.9
   9  2000-01-01 14:00:11     403374       24575     398       28.0%    81.0
  10  2000-07-27 14:57:22     221755       13675     216       28.0%    60.1
[expedition] 15:13:18  best target for e

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [2]:
x.select_target()

[expedition] 15:19:54  === select_target ===



Select by  [t] top ranking   [s] specific starting time  (blank to cancel):  s


Enter a starting time (year ignored). e.g. '2000-05-30 14:59:59' or '05-30 14:59:59'.


Starting time (blank to cancel):  2000-07-24 14:45:55


  -> best target for 2000-07-24 14:45:55: M=288804 ms, F_b=17699, P~27.6%
[expedition] Saved to data/expeditions/metang.json
[expedition] 15:20:06  target set: boot 2000-07-24T14:45:55, timer M=288804 ms, expected F_b=17699 (P~27.6%). Saved.


{'rank': 2170,
 'initial_time': '2000-07-24T14:45:55',
 'M': 288804,
 'target_delay': 17699,
 'second': 284,
 'p': 0.27561597147653105,
 'sigma': 68.55894386587256,
 'mdmsh': [1, 14]}

## Examine target area

In [5]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=288804 ms  ->  mean F_b=17699.0 (target_delay=17699)  sigma=68.6
target second=284  mdmsh(m,h)=(1, 14)  window frames [17459, 17939] (481)
  frame      Δ        seed  hit    weight        w%      cumP%
--------------------------------------------------------------
  17459   -240  0x010E4433    ✗    0.0022    0.001%     0.000%
  17460   -239  0x010E4434    ✓    0.0023    0.001%     0.001%
  17461   -238  0x010E4435    ✗    0.0024    0.001%     0.001%
  17462   -237  0x010E4436    ✗    0.0025    0.001%     0.001%
  17463   -236  0x010E4437    ✗    0.0027    0.002%     0.001%
  17464   -235  0x010E4438    ✗    0.0028    0.002%     0.001%
  17465   -234  0x010E4439    ✓    0.0030    0.002%     0.003%
  17466   -233  0x010E443A    ✓    0.0031    0.002%     0.005%
  17467   -232  0x010E443B    ✗    0.0033    0.002%     0.005%
  17468   -231  0x010E443C    ✗    0.0034    0.002%     0.005%
  17469   -230  0x0

{'p': 0.2756132394900619,
 'n_frames': 481,
 'n_captured': 124,
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [6]:
x.metronome_compass()
x.save()

[expedition] === metronome_compass ===  15:42:35


KeyboardInterrupt: Interrupted by user

## Finding what seed you hit in safari

In [ ]:
x.compass_safari()
x.save()

[expedition] === compass_safari ===  13:43:05
[expedition] Delay from key seed: 24054 frames  (400.90s)
=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit         Metang is angry!
  M / a  Mud, crit (Anger)    Metang is beside itself with anger!
  b      Bait, no crit        Metang is eating!
  B / e  Bait, crit (Eating)  Metang is busy eating!
  0      Ball, 0 shakes       Oh, no! The Pokémon broke free!
  1      Ball, 1 shake        Aww! It appeared to be caught!
  2      Ball, 2 shakes       Aargh! Almost had it!
  3      Ball, 3 shakes       Shoot! It was so close, too!
  C      Captured (ends)      Gotcha! Metang was caught!
  F      Fled (ends)          Metang fled!
  u      Undo last action     —
  ?x     Uncertain result     —
  J      Switch to Jane       —
  Spaces and commas in input are ignored.


Seeds: 482 / 482 remaining
Path:  (none)
Balls: 30
   #        Seed    Delay      Δ
   1. 0x1C1562D1    25297     -1
   2. 0x1D1562D1    25297     -1
   3. 0x1C156

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()